In [ ]:
import random
import cv2

# Define available corruptions (same as config)
_AUG_CTYPES = ["low_light", "gaussian_blur", "motion_blur", "sensor_noise", "fog_haze"]
_AUG_SEVS = ["mild", "moderate", "severe"]

AUG_VISA_ROOT = "/kaggle/working/visa_augmented"

def prepare_augmented_train_data(aug_prob=0.5, rng_seed=0):
    random.seed(rng_seed)
    print(f"Pre-generating augmented training data (aug_prob={aug_prob})...")
    
    # We must only copy train/good. 
    for category in CATEGORIES:
        src_train = os.path.join(VISA_ROOT, category, "train", "good")
        dst_train = os.path.join(AUG_VISA_ROOT, category, "train", "good")
        os.makedirs(dst_train, exist_ok=True)
        
        for fname in sorted(os.listdir(src_train)):
            if not fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")): continue
            dst_path = os.path.join(dst_train, fname)
            
            # Since Kaggle restarts, we can skip if already exists for speed
            if os.path.exists(dst_path): continue
            
            img_bgr = cv2.imread(os.path.join(src_train, fname))
            if img_bgr is None: continue
            img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            
            if random.random() < aug_prob:
                ctype = random.choice(_AUG_CTYPES)
                sev   = random.choice(_AUG_SEVS)
                img = apply_corruption(img, ctype, sev, config, seed=42)
                
            cv2.imwrite(dst_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            
        # Copy test and ground_truth folders as-is
        for subdir in ["test", "ground_truth"]:
            src = os.path.join(VISA_ROOT, category, subdir)
            dst = os.path.join(AUG_VISA_ROOT, category, subdir)
            if os.path.exists(src) and not os.path.exists(dst):
                import shutil
                shutil.copytree(src, dst)
                
    print("Augmented data ready.\n")


In [ ]:
# ---------------------------------------------------------------------------
# 1. Dependency Check
# ---------------------------------------------------------------------------
def ensure_dependencies():
    import subprocess
    import sys
    packages = ["anomalib", "lightning", "albumentationsx", "scikit-image", "opencv-python-headless"]
    for package in packages:
        try:
            check_name = "cv2" if package == "opencv-python-headless" else (
                package.replace("-", "_") if package != "albumentationsx" else "albumentations")
            __import__(check_name)
        except ImportError:
            print(f"Installing missing dependency: {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure_dependencies()

# Now safe to import
import lightning as L
from anomalib.engine import Engine
from anomalib.models import Patchcore, Padim
from anomalib.data import MVTecAD as VisaDataModule
print("Forcing MVTecAD loader for VisA (since Kaggle dataset is in MVTec layout).")
